In [1]:
######################################################
#                        Import                      #
######################################################
import sys
import os
import requests
import importlib
import pandas as pd
from datetime import datetime, timedelta
from config import FRED_API_KEY


In [11]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

# FRED series IDs and their readable names
# Find series IDs at: https://fred.stlouisfed.org
FRED_SERIES = {
    "WPU101707":  "Steel (HR coil)",
    "NATURALGAS": "Natural gas",
    "DCOILWTICO": "Crude oil (WTI)",
    "WPU0911":    "Polymers",
    "PCOPPUSDM":  "Copper",
}

# Unit and currency for each series
SERIES_UNITS = {
    "WPU101707":  ("USD", "per short ton"),
    "NATURALGAS": ("USD", "per MMBtu"),
    "DCOILWTICO": ("USD", "per barrel"),
    "WPU0911":    ("USD", "per metric ton"),
    "PCOPPUSDM":  ("USD", "per metric ton"),
}

# How many years of history to pull
YEARS_BACK = 6

print(f"Configured {len(FRED_SERIES)} FRED series")
print(f"History: {YEARS_BACK} years back from today")


Configured 5 FRED series
History: 6 years back from today


In [12]:
SERIES_UNITS = {
    "WPU101707":  ("USD", "per short ton"),
    "NATURALGAS": ("USD", "per MMBtu"),
    "DCOILWTICO": ("USD", "per barrel"),
    "WPU0911":    ("USD", "per metric ton"),
    "PCOPPUSDM":  ("USD", "per metric ton")
}


#print(SERIES_UNITS)

In [33]:
######################################################
#                      Extract Data                  #
######################################################
def start_date (years_back = YEARS_BACK):
    """
    Argument: 
        years_back(int): Number of years of historic data to pull.

    Return:
        date(date): date of previous date by years_back from today.
    """
    return (datetime.today() - timedelta(365 * years_back)).strftime("%Y-%m-%d")

# start_date(YEARS_BACK)

def end_date():
    return datetime.today().strftime("%Y-%m-%d")

# print(end_date())    

2026-05-26


In [34]:

def get_series_params(series_id, api_key, years_back = YEARS_BACK):
    
    return {
        "series_id":         series_id,
        "api_key":           api_key,
        "file_type":         "json",
        "observation_start": start_date(years_back),
        "observation_end":   end_date(),
    }

# get_series_params("WPU101707",FRED_API_KEY)

{'series_id': 'WPU101707',
 'api_key': '85ab49d89238f68eec5a9cf08315d7b0',
 'file_type': 'json',
 'observation_start': '2020-05-27',
 'observation_end': '2026-05-26'}

In [42]:
def fetch_raw_data(series_id, api_key, years_back=YEARS_BACK):
    """
    Call the FRED API and return the raw response for one series.

    Args:
        series_id (str): FRED series identifier e.g. 'WPU101707'
        api_key (str): Your FRED API key
        years_back (int): Years of history to pull. Default is YEARS_BACK.

    Returns:
        list: Raw list of observation dictionaries from FRED response.
              Each dict contains 'date' and 'value' keys.

    Raises:
        requests.exceptions.HTTPError: If API returns a non-200 status code.
    """
    url    = "https://api.stlouisfed.org/fred/series/observations"
    params = get_series_params(series_id, api_key, years_back)

    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()

    return response.json().get("observations", [])

# Test
# observations = fetch_raw_data("WPU101707", FRED_API_KEY)
# print(f"Rows returned: {len(observations)}")
# print(observations[0])


list